# LH Nautical — Etapa 1: Análise Exploratória de Dados (EDA)

> **Notebook exploratório** — lê os dados brutos (bronze) do Delta Lake.
> As transformações documentadas aqui são executadas automaticamente pelo notebook **02_silver** (pipeline Delta Lake).

**Objetivo:** Inspecionar as quatro bases brutas, identificar problemas de qualidade e documentar as decisões que guiarão o tratamento de dados.

**Tabelas bronze lidas:**
- `bronze_vendas` — histórico de vendas
- `bronze_produtos` — catálogo de produtos
- `bronze_clientes` — dados de clientes
- `bronze_custos` — custos de importação por produto (em **USD**)

> **Dimensão ausente:** custos estão em USD, receitas em BRL. A conversão para calcular margem real exige câmbio histórico — dado **não presente** nas bases brutas. Integrado via API BCB/PTAX pelo notebook 02_silver como tabela `silver_cambio`.

## 0. Setup — Bibliotecas e Caminhos

### Como Executar Este Notebook
1. Execute as células em ordem, do topo ao fim.
2. Dados lidos diretamente do Delta Lake (bronze) — sem dependências locais.
3. Esta etapa é diagnóstica: identifica problemas e define decisões para o pipeline Silver.
4. O checklist final separa diagnóstico do estado bruto e critérios de Go/No-Go.

In [0]:
import pandas as pd
import numpy as np
import ast
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Compatibilidade: Databricks Runtime (UI) vs databricks-connect (VS Code) ─
try:
    spark  # já definido no Databricks runtime
    dbutils.widgets.text("catalog", "workspace",   "Catalog")
    dbutils.widgets.text("schema",  "lh_nautical", "Schema")
    CATALOG = dbutils.widgets.get("catalog")
    SCHEMA  = dbutils.widgets.get("schema")
except NameError:
    from databricks.connect import DatabricksSession
    spark   = DatabricksSession.builder.serverless().getOrCreate()
    CATALOG = "workspace"
    SCHEMA  = "lh_nautical"


df_vendas   = spark.table(f"{CATALOG}.{SCHEMA}.bronze_vendas").toPandas()
df_produtos = spark.table(f"{CATALOG}.{SCHEMA}.bronze_produtos").toPandas()
df_clientes = spark.table(f"{CATALOG}.{SCHEMA}.bronze_clientes").toPandas()
df_custos   = spark.table(f"{CATALOG}.{SCHEMA}.bronze_custos").toPandas()

print('Bases carregadas do Delta Lake!')
print(f'  vendas    : {df_vendas.shape}')
print(f'  produtos  : {df_produtos.shape}')
print(f'  clientes  : {df_clientes.shape}')
print(f'  custos    : {df_custos.shape}')

Bases carregadas do Delta Lake!
  vendas    : (9895, 6)
  produtos  : (157, 4)
  clientes  : (49, 4)
  custos    : (150, 4)


In [0]:
spark.sql("SHOW TABLES IN workspace.lh_nautical").show(truncate=False)

+-----------+--------------------+-----------+
|database   |tableName           |isTemporary|
+-----------+--------------------+-----------+
|lh_nautical|bronze_clientes     |false      |
|lh_nautical|bronze_custos       |false      |
|lh_nautical|bronze_produtos     |false      |
|lh_nautical|bronze_vendas       |false      |
|lh_nautical|gold_fct_vendas     |false      |
|lh_nautical|silver_cambio       |false      |
|lh_nautical|silver_clientes     |false      |
|lh_nautical|silver_custos       |false      |
|lh_nautical|silver_produtos     |false      |
|lh_nautical|silver_vendas       |false      |
|lh_nautical|vwp_kpis_cliente    |false      |
|lh_nautical|vwp_kpis_mensal     |false      |
|lh_nautical|vwp_kpis_produto    |false      |
|lh_nautical|vwp_resumo_executivo|false      |
+-----------+--------------------+-----------+



## 1. Carregamento das Bases

In [0]:
# df_vendas, df_produtos, df_clientes, df_custos ja carregados no setup via spark.table()

## 2. Inspeção Geral de Cada Base

Para cada base vamos verificar:
- Primeiras linhas
- Tipos de dados
- Nulos
- Duplicatas

### 2.1 Vendas

In [0]:
df_vendas.head(10)

,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.00,2023-09-10
1,1,3,136,9,16873.90,15-09-2024
2,2,25,139,7,9475.30,2024-08-13
3,4,20,23,5,55893.00,2023-02-03
4,5,8,57,4,451403.90,2024-02-12
5,6,36,52,3,39056.40,2023-09-26
6,8,27,25,3,34560.05,2024-02-28
7,9,37,26,7,114932.90,07-11-2023
8,10,31,143,3,12643.55,2024-08-25
9,11,39,128,5,23254.00,2023-05-07


In [0]:
df_vendas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9895 entries, 0 to 9894
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          9895 non-null   int32  
 1   id_client   9895 non-null   int32  
 2   id_product  9895 non-null   int32  
 3   qtd         9895 non-null   int32  
 4   total       9895 non-null   float64
 5   sale_date   9895 non-null   object 
dtypes: float64(1), int32(4), object(1)
memory usage: 309.3+ KB


In [0]:
print('=== NULOS ===')
print(df_vendas.isnull().sum())
print(f'\n=== DUPLICATAS ===')
print(f'Linhas duplicadas: {df_vendas.duplicated().sum()}')

# Range do campo id — confirma as lacunas mencionadas no resumo
print(f'\n=== RANGE DO CAMPO id ===')
print(df_vendas['id'].agg(['min', 'max', 'count']))
id_max = df_vendas['id'].max()
print(f'IDs esperados (0..{id_max}): {id_max + 1} | Linhas reais: {len(df_vendas)} | Lacunas: {id_max + 1 - len(df_vendas)}')

# Estatísticas apenas das colunas com significado analítico
print(f'\n=== ESTATÍSTICAS (qtd e total) ===')
display(df_vendas[['qtd', 'total']].describe().round(2))

# Distribuição dos formatos de sale_date
def detectar_formato(d):
    d = str(d)
    if re.match(r'^\d{4}-\d{2}-\d{2}$', d):
        return 'YYYY-MM-DD'
    elif re.match(r'^\d{2}-\d{2}-\d{4}$', d):
        return 'DD-MM-YYYY'
    else:
        return 'outro'

formatos = df_vendas['sale_date'].apply(detectar_formato)
print(f'\n=== FORMATOS DE sale_date ===')
print(formatos.value_counts().to_string())
print(f'\nTotal: {len(df_vendas)} datas — ambos os formatos precisam ser normalizados no tratamento')

=== NULOS ===
id            0
id_client     0
id_product    0
qtd           0
total         0
sale_date     0
dtype: int64

=== DUPLICATAS ===
Linhas duplicadas: 0

=== RANGE DO CAMPO id ===
min         0
max      9999
count    9895
Name: id, dtype: int64
IDs esperados (0..9999): 10000 | Linhas reais: 9895 | Lacunas: 105

=== ESTATÍSTICAS (qtd e total) ===


qtd,total
9895.0,9895.0
8.02,263797.83
4.3,390007.18
1.0,294.5
4.0,23138.2
8.0,82225.0
12.0,339094.5
15.0,2222973.0



=== FORMATOS DE sale_date ===
DD-MM-YYYY    4982
YYYY-MM-DD    4913

Total: 9895 datas — ambos os formatos precisam ser normalizados no tratamento


### 2.2 Produtos

In [0]:
df_produtos.head(10)

,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz
5,Transponder AIS Vector,R$ 11820.21,6,Eletrunicos
6,Radar AIS Zen,R$ 19518.77,7,eLeTrÔnIcOs
7,GPS AIS Zen,R$ 4984.15,8,E L E T R Ô N I C O S
8,Transponder AIS Titan Pulse,R$ 39705.5,9,Eletronicoz
9,Piloto Automático Simrad Titan Flux Magnum,R$ 32033.04,10,eletrônicos


In [0]:
df_produtos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   name             157 non-null    object
 1   price            157 non-null    object
 2   code             157 non-null    int32 
 3   actual_category  157 non-null    object
dtypes: int32(1), object(3)
memory usage: 4.4+ KB


In [0]:
print('=== NULOS ===')
print(df_produtos.isnull().sum())
print(f'\n=== DUPLICATAS ===')
print(f'Linhas duplicadas: {df_produtos.duplicated().sum()}')

# actual_category: lista todas as variações para expor o problema real
print(f'\n=== actual_category — {df_produtos["actual_category"].nunique()} variações únicas (para 3 categorias reais) ===')
print(df_produtos['actual_category'].value_counts().to_string())

=== NULOS ===
name               0
price              0
code               0
actual_category    0
dtype: int64

=== DUPLICATAS ===
Linhas duplicadas: 0

=== actual_category — 39 variações únicas (para 3 categorias reais) ===
AncorageM                9
Ancoraguem               8
Propução                 8
Eletronicoz              7
eletrônicos              7
ELETRONICOS              6
PROPULSAO                6
E L E T R Ô N I C O S    6
P R O P U L S Ã O        6
propulsão                6
Encoragem                5
Propulção                5
Prop                     5
Propulssão               5
aNcOrAgEm                5
A N C O R A G E M        5
Eletrunicos              5
Ancorajm                 5
eLeTrÔnIcOs              5
propulsao                4
Eletrônicos              4
EletrônicoS              3
Ancorajem                3
Ancoragem                3
Propulçao                3
eletronicos              3
AnCoRaGeM                2
Propulsam                2
pRoPuLsÃo          

In [0]:
# Investigação: 157 linhas mas code máximo = 150 (detectado no describe acima)
# Esperamos 150 codes únicos — vamos confirmar
print(f'Total de linhas    : {len(df_produtos)}')
print(f'Codes únicos       : {df_produtos["code"].nunique()}')
print(f'Diferença          : {len(df_produtos) - df_produtos["code"].nunique()} linhas duplicadas por code')
print()
print('Codes com mais de 1 ocorrência:')
print(df_produtos["code"].value_counts()[df_produtos["code"].value_counts() > 1])

Total de linhas    : 157
Codes únicos       : 150
Diferença          : 7 linhas duplicadas por code

Codes com mais de 1 ocorrência:
62     4
145    3
37     2
127    2
Name: code, dtype: int64


### 2.3 Clientes

In [0]:
df_clientes.head(10)

,code,email,full_name,location
0,1,femininos.oliveira.antunes@icloud.com,Femininos Oliveira Antunes,"Aratu (Candeias) , BA"
1,2,nunes.fernanda.soares.azevedo.vieira@outlook.com,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife"
2,3,farias.teixeira.daniel.ribeiro#gmail.com,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS"
3,4,thiago.moreira#gmail.com,Thiago Moreira,"AC , Rio Branco"
4,5,pedro.freitas#icloud.com,Pedro Freitas,PA - Santarém Novo
5,6,coelho.pinheiro.peixoto.antônia.cavalcanti@aol...,Antônia Coelho Pinheiro Peixoto Cavalcanti,"Fortaleza do Tabocão , TO"
6,7,torres.barros.rocha.bianca.siqueira#aol.com,Bianca Barros Rocha Torres Siqueira,PB/Cabedelo
7,8,pimentel.alves.luiz#outlook.com,Luiz Alves Pimentel,SE - Aracaju
8,9,lucas.lopes.guedes.cunha#tutanota.com,Lucas Guedes Cunha Lopes,PB - João Pessoa
9,10,paiva.débora#gmx.com,Débora Paiva,Santarém / PA


In [0]:
df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   code       49 non-null     int64 
 1   email      49 non-null     object
 2   full_name  49 non-null     object
 3   location   49 non-null     object
dtypes: int64(1), object(3)
memory usage: 1.7+ KB


In [0]:
print('=== NULOS ===')
print(df_clientes.isnull().sum())
print(f'\n=== DUPLICATAS ===')
print(f'Linhas duplicadas: {df_clientes.duplicated().sum()}')

# Emails: conta os inválidos (# no lugar de @)
emails_invalidos = df_clientes[~df_clientes['email'].str.contains('@', na=False)]
print(f'\n=== EMAILS ===')
print(f'Emails inválidos (# no lugar de @): {len(emails_invalidos)}/{len(df_clientes)}')

# Location: mostra diversidade de formatos
print(f'\n=== LOCATION — formatos variados (amostra) ===')
print(df_clientes['location'].head(10).to_string())
print('\nPadrões identificados: "UF , Cidade", "Cidade/UF", "UF - Cidade", "Cidade,UF" — sem padronização')

=== NULOS ===
code         0
email        0
full_name    0
location     0
dtype: int64

=== DUPLICATAS ===
Linhas duplicadas: 0

=== EMAILS ===
Emails inválidos (# no lugar de @): 30/49

=== LOCATION — formatos variados (amostra) ===
0        Aratu (Candeias) , BA
1                  PE , Recife
2                Rio Grande,RS
3              AC , Rio Branco
4           PA - Santarém Novo
5    Fortaleza do Tabocão , TO
6                  PB/Cabedelo
7                 SE - Aracaju
8             PB - João Pessoa
9                Santarém / PA

Padrões identificados: "UF , Cidade", "Cidade/UF", "UF - Cidade", "Cidade,UF" — sem padronização


### 2.4 Custos de Importação

In [0]:
df_custos.head(10)

,category,historic_data,product_id,product_name
0,eletrônicos,"[{'start_date': '10/08/2016', 'usd_price': 105...",1,Transponder AIS Maré Magnum
1,eletrônicos,"[{'start_date': '23/11/2017', 'usd_price': 432...",2,Transponder Furuno Marlin
2,eletrônicos,"[{'start_date': '12/04/2016', 'usd_price': 254...",3,Radar Furuno Pulse Leviathan
3,eletrônicos,"[{'start_date': '04/03/2016', 'usd_price': 909...",4,Rádio AIS Hydro Tidal Zen
4,eletrônicos,"[{'start_date': '10/02/2016', 'usd_price': 600...",5,Piloto Automático Furuno Storm
5,eletrônicos,"[{'start_date': '23/07/2020', 'usd_price': 228...",6,Transponder AIS Vector
6,eletrônicos,"[{'start_date': '26/10/2016', 'usd_price': 625...",7,Radar AIS Zen
7,eletrônicos,"[{'start_date': '06/12/2019', 'usd_price': 119...",8,GPS AIS Zen
8,eletrônicos,"[{'start_date': '26/12/2018', 'usd_price': 101...",9,Transponder AIS Titan Pulse
9,eletrônicos,"[{'start_date': '29/05/2018', 'usd_price': 859...",10,Piloto Automático Simrad Titan Flux Magnum


In [0]:
df_custos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   category       150 non-null    object
 1   historic_data  150 non-null    object
 2   product_id     150 non-null    int64 
 3   product_name   150 non-null    object
dtypes: int64(1), object(3)
memory usage: 4.8+ KB


In [0]:
print('=== NULOS ===')
print(df_custos.isnull().sum())
print(f'\n=== DUPLICATAS ===')
cols_hashable = ['product_id', 'product_name', 'category']
print(f'Linhas duplicadas: {df_custos.duplicated(subset=cols_hashable).sum()}')

# Explora a estrutura aninhada de historic_data
print(f'\n=== HISTORIC_DATA — estrutura interna ===')
def parse_hd(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return list(x)  # numpy array ou list ja parseado

primeiro = parse_hd(df_custos['historic_data'].iloc[0])

print(f'Tipo       : {type(primeiro).__name__} de dicts')
print(f'Campos     : {list(primeiro[0].keys())}')
print(f'1º período : {primeiro[0]}')
print(f'Últ período: {primeiro[-1]}')

# Quantos períodos de preço por produto
n_periodos = df_custos['historic_data'].apply(
    lambda x: len(parse_hd(x))
)
print(f'\nPeríodos de preço por produto:')
print(f'  Mínimo : {n_periodos.min()}')
print(f'  Máximo : {n_periodos.max()}')
print(f'  Média  : {n_periodos.mean():.1f}')
print(f'  Total de linhas após explosão (unnest): ~{n_periodos.sum()}')
print(f'\nAção necessária: explodir historic_data em linhas para cruzar com vendas por data de vigência')

=== NULOS ===
category         0
historic_data    0
product_id       0
product_name     0
dtype: int64

=== DUPLICATAS ===
Linhas duplicadas: 0

=== HISTORIC_DATA — estrutura interna ===
Tipo       : list de dicts
Campos     : ['start_date', 'usd_price']
1º período : {'start_date': '10/08/2016', 'usd_price': 10583.63}
Últ período: {'start_date': '08/04/2025', 'usd_price': 5579.75}

Períodos de preço por produto:
  Mínimo : 3
  Máximo : 15
  Média  : 8.4
  Total de linhas após explosão (unnest): ~1260

Ação necessária: explodir historic_data em linhas para cruzar com vendas por data de vigência


## 3. Integridade Referencial entre Bases

Verificamos se todos os IDs nas bases transacionais (`vendas`) têm correspondência nas bases de referência (`clientes`, `produtos`, `custos`). Um ID órfão invalidaria qualquer cruzamento posterior.

In [0]:
# Verifica se todos os IDs em vendas existem nas bases de referência

clientes_em_vendas   = set(df_vendas['id_client'].unique())
clientes_cadastrados = set(df_clientes['code'].unique())
orfaos_cli = clientes_em_vendas - clientes_cadastrados
print(f'id_client em vendas  : {len(clientes_em_vendas)} únicos')
print(f'codes em clientes    : {len(clientes_cadastrados)}')
print(f'Órfãos (sem cadastro): {len(orfaos_cli)} → {"nenhum ✓" if not orfaos_cli else orfaos_cli}')

print()
produtos_em_vendas   = set(df_vendas['id_product'].unique())
produtos_no_catalogo = set(df_produtos['code'].unique())
orfaos_prod = produtos_em_vendas - produtos_no_catalogo
print(f'id_product em vendas : {len(produtos_em_vendas)} únicos')
print(f'codes em produtos    : {len(produtos_no_catalogo)}')
print(f'Órfãos (sem cadastro): {len(orfaos_prod)} → {"nenhum ✓" if not orfaos_prod else orfaos_prod}')

print()
custos_cadastrados = set(df_custos['product_id'].unique())
sem_custo = produtos_em_vendas - custos_cadastrados
print(f'Produtos vendidos sem custo cadastrado: {len(sem_custo)} → {"nenhum ✓" if not sem_custo else sem_custo}')

id_client em vendas  : 49 únicos
codes em clientes    : 49
Órfãos (sem cadastro): 0 → nenhum ✓

id_product em vendas : 150 únicos
codes em produtos    : 150
Órfãos (sem cadastro): 0 → nenhum ✓

Produtos vendidos sem custo cadastrado: 0 → nenhum ✓


## 4. Resumo dos Problemas Encontrados

### Vendas (bronze_vendas) — 9.895 linhas, 6 colunas
| Problema | Evidência | Ação necessária |
|---|---|---|
| `sale_date` em formato misto | **4.982** DD-MM-YYYY e **4.913** YYYY-MM-DD — quase 50/50 | Normalizar para `datetime64` com detecção dinâmica de formato |
| IDs com lacunas | `id` vai de 0 a 9999 mas há 9.895 linhas | Não é erro — indica registros deletados historicamente |
| Valores extremos em `total` | P75 = R$ 339K, máx = R$ 2.2M | Manter — produtos náuticos de alto valor são plausíveis |

---

### Produtos (bronze_produtos) — 157 linhas, 4 colunas
| Problema | Evidência | Ação necessária |
|---|---|---|
| `price` como string | Valores com prefixo `"R$ "` — dtype `object` | Remover prefixo e converter para `float` |
| `actual_category` inconsistente | **39 variações** para apenas 3 categorias reais | Normalizar para categorias canônicas |
| 157 linhas com 150 codes únicos | codes 62 (4×), 145 (3×), 37 (2×), 127 (2×) duplicados | Deduplicar mantendo o primeiro registro |

---

### Clientes (`clientes_crm.json`) — 49 linhas, 4 colunas
| Problema | Evidência | Ação necessária |
|---|---|---|
| Emails com `#` no lugar de `@` | **30 de 49** registros afetados | Substituir `#` por `@` |
| `location` sem padrão | 4+ formatos: `"PE , Recife"`, `"PB/Cabedelo"`, `"PA - Santarém Novo"`, `"Rio Grande,RS"` | Extrair `city` e `state` separadamente |
| Nomes suspeitos | `"Femininos Oliveira Antunes"` — palavra genérica como primeiro nome | Sinalizar com flag, não deletar sem validação |

---

### Custos (`custos_importacao.json`) — 150 linhas, 4 colunas
| Problema | Evidência | Ação necessária |
|---|---|---|
| `historic_data` aninhado | Lista de dicts `{start_date, usd_price}` — **3 a 15 períodos** por produto (~1.260 linhas após explosão) | Explodir para formato longo (uma linha por período) |
| `start_date` como string | Formato `"10/08/2016"` dentro dos dicts | Converter para `datetime64` após explosão |
| Custos em **USD** | Receitas em BRL — conversão impossível sem câmbio histórico | Integrar taxa BCB/PTAX na Etapa 2 |

---

### Câmbio — Dimensão Ausente nas Bases Brutas
| Observação | Evidência | Ação necessária |
|---|---|---|
| Nenhuma base contém taxa de câmbio | Custos em USD, receitas em BRL — sem conversão não há margem real | Integrar câmbio histórico via API BCB/PTAX (Etapa 2) |
| Impacto crítico | Sem câmbio, análise de custo vs receita é **impossível** | 02_silver gera a tabela `silver_cambio` no Delta Lake como 5ª base |

---

### Integridade Referencial entre Bases — ✓ Sem problemas

| Verificação | Resultado |
|---|---|
| `id_client` em vendas ↔ `code` em clientes | **0 órfãos** — 49/49 clientes com cadastro ✓ |
| `id_product` em vendas ↔ `code` em produtos | **0 órfãos** — 150/150 produtos com cadastro ✓ |
| `id_product` em vendas ↔ `product_id` em custos | **0 produtos** sem custo cadastrado ✓ |

## 5. Data Contract Inicial (ponte para o tratamento)

Este contrato define as expectativas minimas de tipo, completude e regra de negocio para cada coluna critica.
Ele sera usado como referencia de validacao na Etapa 2 (`02_tratamento.ipynb`).

In [0]:
data_contract = pd.DataFrame([
    {
        'base': 'vendas',
        'coluna': 'sale_date',
        'tipo_esperado': 'datetime64[ns]',
        'regra': 'Data valida e coerente no periodo 2023-2024',
        'nulos_tolerados': 0
    },
    {
        'base': 'vendas',
        'coluna': 'id_client',
        'tipo_esperado': 'int',
        'regra': 'Deve existir em clientes.code',
        'nulos_tolerados': 0
    },
    {
        'base': 'vendas',
        'coluna': 'id_product',
        'tipo_esperado': 'int',
        'regra': 'Deve existir em produtos.code e custos.product_id',
        'nulos_tolerados': 0
    },
    {
        'base': 'vendas',
        'coluna': 'total',
        'tipo_esperado': 'float',
        'regra': 'Valor monetario positivo',
        'nulos_tolerados': 0
    },
    {
        'base': 'produtos',
        'coluna': 'code',
        'tipo_esperado': 'int',
        'regra': 'Chave unica apos deduplicacao',
        'nulos_tolerados': 0
    },
    {
        'base': 'produtos',
        'coluna': 'price',
        'tipo_esperado': 'float',
        'regra': 'Remover prefixo R$ e converter',
        'nulos_tolerados': 0
    },
    {
        'base': 'produtos',
        'coluna': 'actual_category',
        'tipo_esperado': 'string',
        'regra': 'Padronizar para categorias canonicas',
        'nulos_tolerados': 0
    },
    {
        'base': 'clientes',
        'coluna': 'email',
        'tipo_esperado': 'string',
        'regra': 'Deve conter @ apos correcao',
        'nulos_tolerados': 0
    },
    {
        'base': 'clientes',
        'coluna': 'location',
        'tipo_esperado': 'string',
        'regra': 'Extrair city/state em colunas separadas',
        'nulos_tolerados': 0
    },
    {
        'base': 'custos',
        'coluna': 'historic_data.start_date',
        'tipo_esperado': 'datetime64[ns]',
        'regra': 'Explodir lista e converter data',
        'nulos_tolerados': 0
    },
    {
        'base': 'custos',
        'coluna': 'historic_data.usd_price',
        'tipo_esperado': 'float',
        'regra': 'Explodir lista e converter para numerico',
        'nulos_tolerados': 0
    },
])

print('=== DATA CONTRACT INICIAL ===')
display(data_contract)

=== DATA CONTRACT INICIAL ===


base,coluna,tipo_esperado,regra,nulos_tolerados
vendas,sale_date,datetime64[ns],Data valida e coerente no periodo 2023-2024,0
vendas,id_client,int,Deve existir em clientes.code,0
vendas,id_product,int,Deve existir em produtos.code e custos.product_id,0
vendas,total,float,Valor monetario positivo,0
produtos,code,int,Chave unica apos deduplicacao,0
produtos,price,float,Remover prefixo R$ e converter,0
produtos,actual_category,string,Padronizar para categorias canonicas,0
clientes,email,string,Deve conter @ apos correcao,0
clientes,location,string,Extrair city/state em colunas separadas,0
custos,historic_data.start_date,datetime64[ns],Explodir lista e converter data,0


## 6. Matriz de Decisoes de Limpeza (EDA -> Tratamento)

Cada decisao abaixo inclui risco e validacao esperada na Etapa 2 para evitar ajustes sem rastreabilidade.

In [0]:
duplicated_codes = df_produtos[df_produtos.duplicated(subset=['code'], keep=False)]['code'].unique()
impacto_receita_duplicados = (
    df_vendas[df_vendas['id_product'].isin(duplicated_codes)]['total'].sum()
    / df_vendas['total'].sum() * 100
)

formatos_local = df_vendas['sale_date'].astype(str).str.strip()
pct_datas_mistas = (
    formatos_local.str.match(r'^\d{2}-\d{2}-\d{4}$').sum() / len(formatos_local) * 100
)

matriz_decisoes = pd.DataFrame([
    {
        'problema': 'sale_date em formato misto',
        'decisao_eda': 'Normalizar para datetime com parser de multiplos formatos',
        'risco': 'Baixo (com teste de NaT e range)',
        'validacao_etapa_2': 'sale_date sem NaT e no periodo esperado'
    },
    {
        'problema': 'actual_category com variacoes',
        'decisao_eda': 'Padronizar para categorias canonicas',
        'risco': 'Medio (mapeamento pode forcar classe errada)',
        'validacao_etapa_2': 'Contagem final por categoria e lista de nao mapeados'
    },
    {
        'problema': 'Produtos duplicados por code',
        'decisao_eda': 'Deduplicar por code mantendo primeiro registro',
        'risco': f"Medio (produtos afetados por {impacto_receita_duplicados:.1f}% da receita)",
        'validacao_etapa_2': '150 codes unicos e log de removidos'
    },
    {
        'problema': 'Emails com # no lugar de @',
        'decisao_eda': 'Substituir # por @ quando necessario',
        'risco': 'Baixo (regra deterministica)',
        'validacao_etapa_2': '0 emails invalidos apos correcao'
    },
    {
        'problema': 'historic_data aninhado em custos',
        'decisao_eda': 'Explodir para formato longo',
        'risco': 'Baixo (transformacao estrutural controlada)',
        'validacao_etapa_2': 'shape esperado e tipos convertidos'
    },
    {
        'problema': 'Custo em USD sem cambio historico',
        'decisao_eda': 'Integrar PTAX/BCB diario com preenchimento de feriados',
        'risco': 'Medio (drift se artefato for regenerado sem controle)',
        'validacao_etapa_2': 'cambio_clean sem nulos e periodo completo 2023-2024'
    },
])

print('=== MATRIZ DE DECISOES (EDA -> ETAPA 2) ===')
display(matriz_decisoes)
print(f'\nIndicador de risco adicional: {pct_datas_mistas:.1f}% das datas estavam em DD-MM-YYYY.')

=== MATRIZ DE DECISOES (EDA -> ETAPA 2) ===


problema,decisao_eda,risco,validacao_etapa_2
sale_date em formato misto,Normalizar para datetime com parser de multiplos formatos,Baixo (com teste de NaT e range),sale_date sem NaT e no periodo esperado
actual_category com variacoes,Padronizar para categorias canonicas,Medio (mapeamento pode forcar classe errada),Contagem final por categoria e lista de nao mapeados
Produtos duplicados por code,Deduplicar por code mantendo primeiro registro,Medio (produtos afetados por 2.0% da receita),150 codes unicos e log de removidos
Emails com # no lugar de @,Substituir # por @ quando necessario,Baixo (regra deterministica),0 emails invalidos apos correcao
historic_data aninhado em custos,Explodir para formato longo,Baixo (transformacao estrutural controlada),shape esperado e tipos convertidos
Custo em USD sem cambio historico,Integrar PTAX/BCB diario com preenchimento de feriados,Medio (drift se artefato for regenerado sem controle),cambio_clean sem nulos e periodo completo 2023-2024



Indicador de risco adicional: 50.3% das datas estavam em DD-MM-YYYY.


## 7. Checklist Go/No-Go para Etapa 2

A Etapa 2 so deve iniciar quando os checks criticos estiverem aprovados.

In [0]:
vendas_nulos_criticos = int(df_vendas[['id_client', 'id_product', 'sale_date', 'total']].isna().sum().sum())
produtos_nulos_criticos = int(df_produtos[['code', 'price']].isna().sum().sum())
clientes_nulos_criticos = int(df_clientes[['code', 'email']].isna().sum().sum())
custos_nulos_criticos = int(df_custos[['product_id', 'historic_data']].isna().sum().sum())

emails_invalidos_qtd = int((~df_clientes['email'].str.contains('@', na=False)).sum())
tem_orfaos = len(orfaos_cli) > 0 or len(orfaos_prod) > 0 or len(sem_custo) > 0

checklist = pd.DataFrame([
    {'check': 'Tabelas Delta Lake carregadas', 'tipo': 'go_no_go', 'status': len(df_vendas) > 0},
    {'check': 'Sem orfaos referenciais criticos', 'tipo': 'go_no_go', 'status': not tem_orfaos},
    {'check': 'Nulos criticos em vendas = 0', 'tipo': 'go_no_go', 'status': vendas_nulos_criticos == 0},
    {'check': 'Nulos criticos em produtos = 0', 'tipo': 'go_no_go', 'status': produtos_nulos_criticos == 0},
    {'check': 'Nulos criticos em clientes = 0', 'tipo': 'go_no_go', 'status': clientes_nulos_criticos == 0},
    {'check': 'Nulos criticos em custos = 0', 'tipo': 'go_no_go', 'status': custos_nulos_criticos == 0},
    {'check': 'Custo historico em estrutura aninhada mapeado', 'tipo': 'go_no_go', 'status': 'historic_data' in df_custos.columns},

    # Diagnostico do estado bruto (esperado encontrar problema para tratar na Etapa 2)
    {'check': 'Diagnostico: existem emails invalidos no bruto', 'tipo': 'diagnostico', 'status': emails_invalidos_qtd > 0},
    {'check': 'Diagnostico: sale_date esta em formato misto', 'tipo': 'diagnostico', 'status': formatos.nunique() > 1},
])

checklist['resultado'] = checklist['status'].map({True: 'PASS', False: 'FAIL'})
display(checklist[['tipo', 'check', 'resultado']])

check_go_no_go = checklist[checklist['tipo'] == 'go_no_go']['status'].all()
if check_go_no_go:
    print('\nGO: EDA pronta para avancar para a Etapa 2.')
else:
    print('\nNO-GO: revise os itens críticos com FAIL antes de seguir.')

print(f"\nDiagnóstico bruto: emails inválidos identificados = {emails_invalidos_qtd} (esperado > 0 nesta etapa).")

tipo,check,resultado
go_no_go,Tabelas Delta Lake carregadas,PASS
go_no_go,Sem orfaos referenciais criticos,PASS
go_no_go,Nulos criticos em vendas = 0,PASS
go_no_go,Nulos criticos em produtos = 0,PASS
go_no_go,Nulos criticos em clientes = 0,PASS
go_no_go,Nulos criticos em custos = 0,PASS
go_no_go,Custo historico em estrutura aninhada mapeado,PASS
diagnostico,Diagnostico: existem emails invalidos no bruto,PASS
diagnostico,Diagnostico: sale_date esta em formato misto,PASS



GO: EDA pronta para avancar para a Etapa 2.

Diagnóstico bruto: emails inválidos identificados = 30 (esperado > 0 nesta etapa).


## 8. Resumo Executivo (Etapa 1)

- A EDA confirmou problemas de qualidade reais em vendas, produtos, clientes e custos.
- Não há quebra de integridade referencial entre vendas, clientes, produtos e custos.
- O maior risco de análise de margem no bruto é a ausência de câmbio histórico.
- As decisões de limpeza foram documentadas com risco e validação esperada na Etapa 2.
- O Go/No-Go agora separa critérios críticos de execução e diagnóstico do estado bruto.

In [0]:
def run_all_quality_checks_eda(df_vendas_local, df_produtos_local, df_clientes_local, df_custos_local):
    """Executa checks críticos da Etapa 1 para facilitar reexecução controlada."""
    checks = {
        'vendas_colunas_criticas_ok': set(['id_client', 'id_product', 'sale_date', 'total']).issubset(df_vendas_local.columns),
        'produtos_colunas_criticas_ok': set(['code', 'price', 'actual_category']).issubset(df_produtos_local.columns),
        'clientes_colunas_criticas_ok': set(['code', 'email', 'location']).issubset(df_clientes_local.columns),
        'custos_colunas_criticas_ok': set(['product_id', 'historic_data']).issubset(df_custos_local.columns),
        'integridade_clientes_sem_orfaos': len(set(df_vendas_local['id_client']) - set(df_clientes_local['code'])) == 0,
        'integridade_produtos_sem_orfaos': len(set(df_vendas_local['id_product']) - set(df_produtos_local['code'])) == 0,
    }
    out = pd.DataFrame({'check': checks.keys(), 'status': checks.values()})
    out['resultado'] = out['status'].map({True: 'PASS', False: 'FAIL'})
    return out

qa_eda = run_all_quality_checks_eda(df_vendas, df_produtos, df_clientes, df_custos)
display(qa_eda[['check', 'resultado']])

if qa_eda['status'].all():
    print('RUN_ALL CHECKS (EDA): PASS')
else:
    print('RUN_ALL CHECKS (EDA): FAIL')

check,resultado
vendas_colunas_criticas_ok,PASS
produtos_colunas_criticas_ok,PASS
clientes_colunas_criticas_ok,PASS
custos_colunas_criticas_ok,PASS
integridade_clientes_sem_orfaos,PASS
integridade_produtos_sem_orfaos,PASS


RUN_ALL CHECKS (EDA): PASS
